# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of BabyAGI, but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information.

## Install and Import Required Modules

In [1]:
!pip install langchain_community langchain_experimental langchain_openai langchain_classic
!pip install faiss-cpu google-search-results python-dotenv


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [2]:
from typing import Optional, List, Any

from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_experimental.autonomous_agents import BabyAGI

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [3]:
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

In [4]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
SERPAPI_API_KEY = os.getenv('SERPAPI_API_KEY')

In [5]:
# Define your embedding model
embeddings_model = OpenAIEmbeddings()

# Initialize the vectorstore as empty
import faiss

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(
    embedding_function=embeddings_model,
    index=index,
    docstore=InMemoryDocstore({}),
    index_to_docstore_id={}
)

## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks

NOTE: in this notebook, the Execution chain will now be an agent.

In [6]:
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_classic.chains.base import Chain
from langchain_core.tools import Tool
from langchain_community.utilities import SerpAPIWrapper

# LLM
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# TODO tool using LLM
todo_prompt = PromptTemplate.from_template(
    "You are a planner. Come up with a todo list for this objective: {input}"
)
todo_chain = todo_prompt | llm

# Search tool via SerpAPI
search = SerpAPIWrapper()

tools = [
    Tool(
        name="Search",
        func=search.run,
        description="Use this FIRST to get real current weather data. Input: a search query like 'San Francisco weather today'.",
    ),
    Tool(
        name="TODO",
        func=lambda objective: todo_chain.invoke({"input": objective}).content,
        description="Use this to plan steps. Only use AFTER searching for real data.",
    ),
]

# ReAct prompt (no hub needed)
react_prompt = PromptTemplate.from_template(
    "You are an AI assistant. Your goal is: {input}\n\n"
    "You have access to these tools:\n{tools}\n\n"
    "IMPORTANT: Always use Search FIRST to get real current data before anything else.\n\n"
    "Format:\n"
    "Thought: what to do\n"
    "Action: tool name (one of [{tool_names}])\n"
    "Action Input: input for the tool\n"
    "Observation: result\n"
    "... (repeat as needed)\n"
    "Thought: I now know the final answer\n"
    "Final Answer: the complete answer\n\n"
    "Begin!\n\n"
    "Thought:{agent_scratchpad}"
)

# Create ReAct agent
react_agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=react_prompt,
)

# AgentExecutor
agent_executor = AgentExecutor(
    agent=react_agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
)

# Wrapper Chain so BabyAGI's internal .run() works correctly
class AgentExecutorChain(Chain):
    executor: AgentExecutor

    @property
    def input_keys(self) -> List[str]:
        return ["objective", "task", "context"]

    @property
    def output_keys(self) -> List[str]:
        return ["output"]

    def _call(self, inputs: dict) -> dict:
        prompt = (
            f"Objective: {inputs['objective']}\n"
            f"Context: {inputs['context']}\n"
            f"Task: {inputs['task']}"
        )
        result = self.executor.invoke({"input": prompt})
        return {"output": result.get("output", "")}

### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [7]:
OBJECTIVE = "Write a weather report for SF today"

In [8]:
import uuid
from langchain_experimental.autonomous_agents.baby_agi import baby_agi as baby_agi_module

# Fix: BabyAGI reuses the same result IDs across runs, causing a duplicate key error.
# This patch replaces the fixed IDs with unique UUIDs on every add_texts call.
original_call = baby_agi_module.BabyAGI._call

def patched_call(self, inputs, run_manager=None):
    original_add_texts = self.vectorstore.add_texts
    def unique_add_texts(texts, metadatas=None, ids=None, **kwargs):
        ids = [str(uuid.uuid4()) for _ in texts]
        return original_add_texts(texts, metadatas=metadatas, ids=ids, **kwargs)
    self.vectorstore.add_texts = unique_add_texts
    return original_call(self, inputs, run_manager=run_manager)

baby_agi_module.BabyAGI._call = patched_call

# Logging
verbose = False
# If None, will keep on going forever
max_iterations: Optional[int] = 3

baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=AgentExecutorChain(executor=agent_executor),
    verbose=verbose,
    max_iterations=max_iterations,
)

In [9]:
baby_agi.invoke({"objective": OBJECTIVE})


*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


> Entering new AgentExecutor chain...
I need to search for the current weather in San Francisco today.
Action: Search
Action Input: 'San Francisco weather today'{'type': 'weather_result', 'temperature': '55', 'unit': 'Fahrenheit', 'precipitation': '1%', 'humidity': '90%', 'wind': '13 mph', 'location': 'San Francisco, CA', 'date': 'Wednesday 6:00 AM', 'weather': 'Mostly sunny'}I have the current weather data for San Francisco today. Now I need to create a weather report and make a todo list.
Action: TODO
Action Input: Write a weather report for SF today1. Check the current weather conditions in San Francisco
2. Research any potential weather patterns or changes expected for the day
3. Gather any relevant data or information needed for the report
4. Write a detailed description of the current weather conditions in San Francisco
5. Include any important information such as temperature, humidity, wind spe

{'objective': 'Write a weather report for SF today'}